# 12 - Wind what-if scenarios (digital-twin scenario engine)

Precomputes WA-TGCN v2 forecasts under perturbed wind for the 300 dashboard
playback windows, so `app.py` can offer a wind-scenario slider without live
torch inference.

Scenarios (applied to the 48 h input wind, and to A(t) recomputed from it;
everything else - PM2.5 history, calendar, anchor, forecast climatology graph -
held fixed):

| key | meaning |
|---|---|
| `baseline` | unchanged (sanity check vs playback.json) |
| `wind_x2` | wind speed doubled |
| `wind_calm` | wind speed x0.2 (near-stagnation) |
| `wind_nw` | wind rotated to blow from the north-west (stubble-burning inflow), speed kept |

Output: `dashboard_export/whatif.json` (~4 MB). Nothing else touched.

In [25]:
import os, json, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR = os.path.join(OUT, 'scalers')
MODEL_DIR  = os.path.join(OUT, 'models')
DASH       = os.path.join(V2_DIR, 'dashboard_export')

DATASET_CSV      = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
DYNAMIC_ADJ_PATH = os.path.join(V2_DIR, 'dynamic_adj_matrix.npy')
STATIONS_EXCEL   = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
CLIM_GRID_PATH   = os.path.join(V2_DIR, 'v2_clim_grid.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
THRESHOLD_KM = 12.0
CPCB_THRESHOLD = 60.0

cfg = json.load(open(os.path.join(MODEL_DIR, 'v2_config.json')))
FEATURE_COLS = cfg['feature_cols']; TIME_COLS = cfg['time_cols']
F = cfg['in_features']; SEEDS = cfg['seeds']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
print(f'Device {DEVICE} | F={F} | {len(SEEDS)} seeds')

Device cuda | F=26 | 7 seeds


In [26]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
N = len(station_names)
timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
dyn_adj = np.load(DYNAMIC_ADJ_PATH).astype(np.float32)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a))
dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
within_threshold = (dist_matrix > 0) & (dist_matrix <= THRESHOLD_KM)
dir_u = np.zeros((N, N)); dir_v = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            de = (lons[j]-lons[i]) * np.cos(np.radians(lats[i]))
            dn = lats[j]-lats[i]; nr = np.hypot(de, dn)
            if nr > 0:
                dir_u[i, j], dir_v[i, j] = de/nr, dn/nr
print(f'N={N} T={T}')

N=37 T=19921


In [27]:
# same 26-feature recipe as 09 / 10 / 11
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at','u_wind','v_wind']
g = df.groupby('Station', sort=False)
df['wspd'] = np.hypot(df['u_wind'], df['v_wind'])
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)
built = (BASE_FEATS +
    ['wspd','pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
assert built == FEATURE_COLS
TIME_IDX = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')

X_raw = np.zeros((T, N, F), dtype=np.float32)                    # UNscaled - wind gets perturbed here
for fi, col in enumerate(FEATURE_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    X_raw[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))
Y_raw = X_raw[:, :, 0].copy()
print('X_raw', X_raw.shape)

X_raw (19921, 37, 26)


In [28]:
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v2_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v2_log_scaler.pkl'), 'rb'))
def scale_X(x):  return ((x.reshape(-1, F) - feat_scaler.mean_) / feat_scaler.scale_).reshape(x.shape).astype(np.float32)
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Yl_full = to_log(Y_raw)

clim_grid = np.load(CLIM_GRID_PATH)
def compute_At_vec(u, v, eps=1e-6):
    al = u[:, None]*dir_u + v[:, None]*dir_v
    A = np.where(within_threshold & (al > 0.0), al/(dist_matrix + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs
mo_arr = pd.DatetimeIndex(timestamps).month.values - 1
ho_arr = pd.DatetimeIndex(timestamps).hour.values
dyn_clim = np.zeros((T, N, N), dtype=np.float32)
for t in range(T):
    uv = clim_grid[mo_arr[t], ho_arr[t]]
    dyn_clim[t] = compute_At_vec(uv[:, 0], uv[:, 1])
print('scalers + climatology graph ready')

scalers + climatology graph ready


In [29]:
class WATGCNv2(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_time=7, gcn_hops=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.time_enc = nn.Sequential(nn.Linear(n_time, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = H, forecast_horizon
    @staticmethod
    def _n(A): return A / A.sum(-1, keepdim=True).clamp(min=1e-6)
    def forward(self, X, At, Afc, TFut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        A = self._n(At + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        t_emb = self.time_enc(TFut).unsqueeze(2).expand(B, self.FH, n, self.H)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, t_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

models = []
for sd in SEEDS:
    m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v2_seed{sd}.pt'), map_location=DEVICE))
    m.eval(); models.append(m)
print(f'loaded {len(models)} v2 checkpoints')

loaded 7 v2 checkpoints


In [30]:
# which windows: the same 300 the dashboard samples
val_end_idx = int(T * 0.85)
n_test = (T - val_end_idx) - (INPUT_WINDOW + HORIZON) + 1
test_g = np.arange(val_end_idx, val_end_idx + n_test)
sample_idx = np.linspace(0, n_test - 1, 970).astype(int)
sample_g = test_g[sample_idx]
now_ts = timestamps[val_end_idx + INPUT_WINDOW : val_end_idx + INPUT_WINDOW + n_test]

def perturb(x48, kind):
    """x48: (48, N, F) unscaled. returns perturbed copy + its per-hour (u, v) arrays (48, N)."""
    x = x48.copy()
    u, v = x[:, :, U_IDX].copy(), x[:, :, V_IDX].copy()
    if kind == 'baseline':
        pass
    elif kind == 'wind_x2':
        u, v = u * 2.0, v * 2.0                     # stronger wind -> more dispersion
    elif kind == 'wind_nw':
        spd = np.hypot(u, v)
        u, v = spd * 0.7071, -spd * 0.7071          # wind FROM NW -> air (stubble smoke) moves SE into Delhi
    elif kind == 'wind_rev':
        u, v = -u, -v                               # flip the airshed - opposite stations become upwind
    x[:, :, U_IDX] = u; x[:, :, V_IDX] = v; x[:, :, W_IDX] = np.hypot(u, v)
    return x, u, v

SCEN = ['baseline', 'wind_x2', 'wind_nw', 'wind_rev']

@torch.no_grad()
def run_scenario(kind):
    preds = np.zeros((len(sample_g), HORIZON, N), dtype=np.float32)
    B = 32
    for b0 in range(0, len(sample_g), B):
        gg = sample_g[b0:b0+B]
        Xb, Ab = [], []
        for g_ in gg:
            x48, u, v = perturb(X_raw[g_:g_+INPUT_WINDOW], kind)
            Xb.append(scale_X(x48))
            Ab.append(np.stack([compute_At_vec(u[h], v[h]) for h in range(INPUT_WINDOW)]))
        Xb  = torch.from_numpy(np.stack(Xb)).to(DEVICE)
        Ab  = torch.from_numpy(np.stack(Ab)).to(DEVICE)
        Afc = torch.from_numpy(np.stack([dyn_clim[g_+INPUT_WINDOW:g_+INPUT_WINDOW+HORIZON] for g_ in gg])).to(DEVICE)
        Tf  = torch.from_numpy(np.stack([np.ascontiguousarray(
                  scale_X(X_raw[g_+INPUT_WINDOW:g_+INPUT_WINDOW+HORIZON])[:, 0, :][:, TIME_IDX]) for g_ in gg])).to(DEVICE)
        An  = torch.from_numpy(np.stack([Yl_full[g_+INPUT_WINDOW-1] for g_ in gg])).to(DEVICE)
        ens = torch.stack([mm(Xb, Ab, Afc, Tf, An) for mm in models], 0).mean(0)
        preds[b0:b0+len(gg)] = ens.cpu().numpy()
    return from_log(preds)

t0 = time.time()
scen_preds = {k: run_scenario(k) for k in SCEN}
print(f'all scenarios in {time.time()-t0:.0f}s')

# sanity: baseline here vs playback.json predictions
pb = json.load(open(os.path.join(DASH, 'playback.json')))
b_here = scen_preds['baseline'][0, :, 0]
b_json = np.array(pb[0]['stations'][0]['predicted_24h'])
print(f'baseline check (window 0, station 0) max abs diff vs playback.json: {np.abs(b_here - b_json).max():.2f}  (expect < ~1)')

all scenarios in 13s
baseline check (window 0, station 0) max abs diff vs playback.json: 0.05  (expect < ~1)


In [31]:
labels = {'wind_x2': 'Wind doubled', 'wind_nw': 'NW inflow (stubble smoke)', 'wind_rev': 'Wind reversed'}
EXPORT_SCEN = ['wind_x2', 'wind_nw', 'wind_rev']
out = {'scenarios': [{'key': k, 'label': labels[k]} for k in EXPORT_SCEN],
       'cpcb_threshold': CPCB_THRESHOLD, 'windows': {}}
for wi, gi in enumerate(sample_idx):
    e = {}
    for k in EXPORT_SCEN:
        p = scen_preds[k][wi]
        e[k] = {'predicted_24h': np.round(p).astype(int).T.tolist(),
                'peak': np.round(p.max(0)).astype(int).tolist(),
                'breach_h': [int(next((h+1 for h, v in enumerate(p[:, s]) if v > CPCB_THRESHOLD), 0)) for s in range(N)]}
    out['windows'][str(int(gi))] = e
path = os.path.join(DASH, 'whatif.json')
json.dump(out, open(path, 'w'), separators=(',', ':'))
print(f'saved {path}  {os.path.getsize(path)/1e6:.1f} MB  {len(out["windows"])} x {len(EXPORT_SCEN)}')
for k in EXPORT_SCEN:
    print(f'  {labels[k]:26s} {(scen_preds[k]-scen_preds["baseline"]).mean():+.1f} ug/m3')

saved D:\FYP\Eco-Twin\v2_clean\dashboard_export\whatif.json  10.2 MB  970 x 3
  Wind doubled               -2.5 ug/m3
  NW inflow (stubble smoke)  +5.4 ug/m3
  Wind reversed              +2.1 ug/m3
